Fundamentos de Databricks

In [0]:
import pandas as pd


In [0]:
spark.range(1).show

In [0]:
%sql
SELECT current_user() AS usuario,
       current_catalog() AS catalogo,
       current_schema() AS esquema

1. Creación del entorno

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS analitica


In [0]:
%sql
USE CATALOG analitica


1.1 Creación del esquema dentro del catalogo    

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS analitica.data;


In [0]:
%sql
SHOW SCHEMAS IN analitica


1.2 Creación del Volumen

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS analitica.data.landing


2. Creación de tablas con SQL

In [0]:
%sql
USE analitica.data

In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple (
  id INT,
  nombre STRING
)
USING DELTA

### 2.1 Insertar Data

In [0]:
%sql
INSERT INTO tabla_simple VALUES (1, 'Ana'), (2, 'Luis'), (3, 'María')


In [0]:
%sql
SELECT * FROM tabla_simple ORDER BY id


### 3. FUnciones de PySpark

In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/analitica/data/landing"

In [0]:
%sql
CREATE OR REPLACE FUNCTION analitica.data.fn_mayor_que_10(x DOUBLE)
RETURN x > 10


### 4. Lectura de Datos

In [0]:
path_data_demo = "/Volumes/analitica/data/landing/data_demo.csv"
df = spark.read.csv(
    path=path_data_demo,
    header=True,
    inferSchema=True
)


### 4.1 Guardar Tablas

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("Inmuebles")

In [0]:
%sql
SELECT * FROM inmuebles LIMIT 10

In [0]:
path_data_demo = "/Volumes/analitica/data/landing/spark_intro.csv"

### 5.1 Selección y filtro

In [0]:
from pyspark.sql import functions as F

In [0]:
df_pedidos = spark.read.csv(
    path=path_data_demo,
    header=True,
    inferSchema=True
)
display(df_pedidos.limit(10))

In [0]:
display(
    df_pedidos.select("order_id", "country", "product", "units_sold")
    .filter(F.col("country")=="Spain")
)

### 5.2 Calcula Pedidos


In [0]:
df_importes = df_pedidos.withColumn(
    "importe", F.col("units_sold") * F.col("unit_price")
)
display(df_importes)

In [0]:
resumen_pais = df_importes.groupBy("country").agg(
    F.sum("units_sold").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("importe_total")
)
display(resumen_pais.orderBy(F.desc("importe_total")))

### 6. LEctura de las dimensiones

In [0]:
path_data_demo = "/Volumes/analitica/data/landing/dim_spark_intro.csv"
df_productos = spark.read.csv(
    path=path_data_demo,
    header=True,
    schema="product STRING, product_family STRING, supplier STRING",
    mode="FAILFAST"
)
display(df_productos)

### 6.1 Union sencilla

In [0]:
display(df_productos.groupBy("product").count().filter("count > 1"))

df_enriquecido = df_importes.join(df_productos, "product", "left")
display(df_enriquecido)

print("Pedidos antes:", df_pedidos.count())
print("Pedidos después:", df_enriquecido.count())

display(df_pedidos.join(df_productos, "product", "left_anti"))

assert df_enriquecido.count() == df_pedidos.count(), "Revisar la llave de productos."

In [0]:
df_enriquecido.write.format("delta").mode("overwrite").saveAsTable("pedidos_enriquecidos")

### 7. Lectura del ejemplo de ventas

In [0]:
path_data_demo = "/Volumes/analitica/data/landing/ventas_2025.csv"
df_ventas = spark.read.csv(
    path=path_data_demo,
    header=True,
    schema="id INT, fecha DATE, producto STRING, cantidad INT, precio DOUBLE",
    dateFormat="yyyy-MM-dd",
    mode="FAILFAST"
)
display(df_ventas.limit(10))
df_ventas.printSchema()
print("Registros de venta:", df_ventas.count())

In [0]:
display(df_ventas.groupBy("id").count().filter("count > 1"))

df_invalidos = df_ventas.filter(
    F.col("id").isNull() | F.col("fecha").isNull() |
    F.col("producto").isNull() | F.col("cantidad").isNull() |
    F.col("precio").isNull() | F.isnan("precio") |
    (F.col("cantidad") <= 0) | (F.col("precio") < 0)
)
display(df_invalidos)
assert df_invalidos.count() == 0, "Revisar los registros antes de continuar."

### 7.2 Tabla Delta de ventas

In [0]:
df_ventas.write.format("delta").mode("overwrite").saveAsTable("ventas")

In [0]:
%sql
DESCRIBE DETAIL ventas

### 7.3 Consulta del importe por producto

In [0]:
%sql
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto
ORDER BY importe_total DESC

Visualización: sobre la salida anterior, **+ → Visualization → Bar**, eje X `producto`, eje Y `importe_total`, guardar como *Importe por producto · muestra de ventas 2025*.

### 7.4 Una variable para cambiar el filtro

In [0]:
pais_elegido = "Spain"
display(df_enriquecido.filter(F.col("country") == pais_elegido))

### 8. Operaciones Delta

### 8.1 Copia de ensayo

In [0]:
%sql
CREATE OR REPLACE TABLE ventas_delta_demo USING DELTA AS SELECT * FROM ventas

### 8.2 Versión de referencia

In [0]:
historial = spark.sql("DESCRIBE HISTORY ventas_delta_demo")
version_inicial = historial.orderBy(F.desc("version")).first()["version"]
print("Versión de referencia:", version_inicial)

### 8.3 UPDATE

In [0]:
%sql
UPDATE ventas_delta_demo SET precio = ROUND(precio + 1, 2) WHERE id = 1

In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id = 1

### 8.4 INSERT

In [0]:
%sql
INSERT INTO ventas_delta_demo VALUES (1001, DATE '2025-01-31', 'Producto demo', 2, 10.0)

In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

### 8.5 DELETE

In [0]:
%sql
DELETE FROM ventas_delta_demo WHERE id = 1001

In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

### 8.6 Fuente de cambios para MERGE

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cambios_ventas AS
SELECT * FROM VALUES
  (1, DATE '2025-01-03', 'Teclado', 20, CAST(27.99 AS DOUBLE)),
  (1001, DATE '2025-01-31', 'Producto demo', 2, CAST(10.0 AS DOUBLE))
AS cambios(id, fecha, producto, cantidad, precio)

In [0]:
%sql
SELECT * FROM cambios_ventas

### 8.7 MERGE

In [0]:
%sql
MERGE INTO ventas_delta_demo AS destino
USING cambios_ventas AS fuente
ON destino.id = fuente.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id IN (1, 1001) ORDER BY id

### 8.8 Historial

In [0]:
%sql
DESCRIBE HISTORY ventas_delta_demo

### 8.9 Time Travel

In [0]:
df_inicial = spark.read.option("versionAsOf", version_inicial).table("ventas_delta_demo")
print("Filas iniciales:", df_inicial.count())
print("Filas actuales:", spark.table("ventas_delta_demo").count())
display(df_inicial.filter(F.col("id") == 1))
display(spark.table("ventas_delta_demo").filter(F.col("id") == 1))

### 8.10 Comprobación

In [0]:
%sql
SELECT 'Pedidos enriquecidos' AS tabla, COUNT(*) AS registros FROM pedidos_enriquecidos
UNION ALL
SELECT 'Ventas base', COUNT(*) FROM ventas
UNION ALL
SELECT 'Ensayo Delta', COUNT(*) FROM ventas_delta_demo

### 9. Vistas, funciones y control de acceso

### 9.1 Creación de una vista

In [0]:
%sql
CREATE OR REPLACE VIEW v_ventas_por_producto AS
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto

In [0]:
%sql
SELECT * FROM v_ventas_por_producto ORDER BY importe_total DESC

### 9.2 Función reutilizable

In [0]:
%sql
CREATE OR REPLACE FUNCTION es_venta_alta(importe DOUBLE)
RETURNS BOOLEAN
RETURN importe > 500

In [0]:
%sql
SELECT id, producto, cantidad, precio,
       es_venta_alta(cantidad * precio) AS supera_umbral
FROM ventas
ORDER BY id
LIMIT 10

### 9.3 Propiedad y permisos

In [0]:
%sql
DESCRIBE TABLE EXTENDED v_ventas_por_producto

In [0]:
%sql
SHOW GRANTS ON VIEW v_ventas_por_producto

### 9.4 Concesión de acceso (descomentar solo si existe el grupo y hay permiso)

In [0]:
%sql
-- GRANT USE CATALOG ON CATALOG analitica TO `lectores_clase`;
-- GRANT USE SCHEMA ON SCHEMA analitica.data TO `lectores_clase`;
-- GRANT SELECT ON VIEW analitica.data.v_ventas_por_producto TO `lectores_clase`;
SHOW GRANTS ON VIEW v_ventas_por_producto;

### 9.5 Prueba con otra identidad (la ejecuta otra persona desde su cuenta)

In [0]:
%sql
SELECT * FROM analitica.data.v_ventas_por_producto

### 9.6 Linaje: Catalog → analitica → data → v_ventas_por_producto → pestaña **Lineage**.